In [1]:
# dev version
%load_ext autoreload

# Pythonic Proces Based Modeling 
Pythonic implementation of ProBMoT. **User-friendly** and **fast**.

# TODO 
Structure of this introduction should follow this plan:

**Structure**

1. **model** has **variables** and **entities** . Model should be the base, populate it with entities and variables. 
2. **equations** - use choose for multiple options, BUILD a model 
3. **entities** and **entity templates**

**Usage**
1. Inducing 
2. Estimating
3. Simulating

## Usage
PyBM implements *Process-based modeling*, formalized in [Process-Based Models of Dynamical Systems - Representation and Induction](http://probmot.ijs.si/pubs/Darko_Cerepnalkoski_PhD.pdf). 

The implementation tries to follow the formal structure of ProBMoT, while making it a little more readable. 

Each model is representent by entities and processes. 

First we show a simple example using physical particles. 
Lets assume that we want to model a particle with mass and coordinates x and y in 2d plane. 

### Variables and constants 

In [2]:
from pybm.model import Var, Const
x  = Var("x")
x.data = 5
x.data

5

blabla

In [3]:
# define variables and constats 
from pybm.model import Entity
import numpy as np

# create a particle with parameters x, y and mass
v1 = Var("x")
v2 = Var("y")
c1 = Const("mass", value=0.5)
particle = Entity(v1, v2, c1, name="particle")

# acces and change its data 
particle["x"] = np.random.rand(1)

# variables and constants can be accessed by their name
print(f"{particle.get('x')} with value {particle["x"]}")  # prints the variable x

x with value [0.34116541]


## Entities and entity templates

Every particle will have this properties. So we can create a new entity template `EntityTemp` which is a placeholder for a general particle. 

Created particles can be edited. New variables and constants can be added using `particle.add()`

In [4]:
from pybm.model import VarTemp, ConstTemp, EntityTemp

particle_template = EntityTemp(
    VarTemp("x"),
    VarTemp("y"),
    ConstTemp("k", value=0.5)
)
# create one instance of the template
neon_particle = particle_template(name="neon_particle")

# create another and add a new variable to it
higgs_boson = particle_template(name="higgs_boson")
higgs_boson.add(Var("spin"))

Every elementary particle has its own spin. Instead of adding it to each elementary particle seperatly, we can create a new particle template and add spin to it.

A new enity , which inherits the structure from another entity, is created with `EntityTemp.inherits(parent_entity)`. 
**DO NOT** create a new entity with `new_entity = old_entity`. This will link `new_entity` to the old one. Changing the new one will affect the old one.

In [5]:

# every elementary particle has its own spin --> create a new template for elementary particles based on the old one 
elementary_particle_template = EntityTemp.inherits(particle_template)
elementary_particle_template.add(VarTemp("spin"))

# now create a new instance of the elementary particle template - it will automaticly have the spin variable
electron = elementary_particle_template(name="electron")

str(electron)

'Entity(x, y, k, spin)'

A list of variables, used in entity template, can be accessed with:

In [6]:
elementary_particle_template.constants(), elementary_particle_template.variables()

(['k'], ['x', 'y', 'spin'])

element

## Models and equations
A model class `Model` is a container of all the variables. Every entity should have an active model, which stores the values of the variables. 

Variables in equations should be referenced ONLY via a model or via an entity. This way, the same code can be used for different model structures. 

In [7]:
from pybm.model import EMPTY_MODEL, Model

model_from_entity = Model(electron)

## Equations
Idea: Equations are (should) be given by 
```
d(var)/dt = f(entity1["var1_name"], entity1["var2_name"], ...., entityN["varM_name"])
```
or as 
```
d(var)/dt = f(vars["var1_name"], vars["var2_name"], )
```
where vars is a "special" entity that includes all the variables. 

Each entity has its own  `self.active_model : Model`, which contains the references to the right variables.

This way, we can represent and equation by a function ` lambda TODO : f(...)` and keep the same syntax lets hope!?

## Example

We will model a simple system with a particle that either moves via formula $x'(t) = -x(t)$ or $x'(t) = 0$. 

In [8]:
from pybm.model import Choose

# create a fresh model and add a simple entity 
model = Model()
particle = Entity(Var("x"), name="particle", active_model=model)

# add equation for the particle 
# right side of the equation: 
def rs_exp(t):
    return particle["x"]

def rs_const(t):
    return 0 

# add a choice for the right side of the equation
particle.get("x").ode = Choose(rs_exp, rs_const)

model.elements, model.vars, model.entities

({'particle': <pybm.model.Entity at 0x7c6581584fc0>,
  'particle.x': Var(name='x', type='not_set', initial=None, range=None, aggregation=None, unit=None, ode=<pybm.model.Choose object at 0x7c65815d97f0>, algebraic=None)},
 {'particle.x': Var(name='x', type='not_set', initial=None, range=None, aggregation=None, unit=None, ode=<pybm.model.Choose object at 0x7c65815d97f0>, algebraic=None)},
 {'particle': <pybm.model.Entity at 0x7c6581584fc0>})

In [9]:
# get all the possible models induced from the current model
induced_models = model.induce()

induced_models

[Model(Entities: [<pybm.model.Entity object at 0x7c65815808d0>], Vars: [Var(name='x', type='not_set', initial=None, range=None, aggregation=None, unit=None, ode=<function rs_exp at 0x7c65815a6820>, algebraic=None)], Consts: []),
 Model(Entities: [<pybm.model.Entity object at 0x7c65815806b0>], Vars: [Var(name='x', type='not_set', initial=None, range=None, aggregation=None, unit=None, ode=<function rs_const at 0x7c65815a6770>, algebraic=None)], Consts: [])]

In [10]:
model = induced_models[0]



In [12]:
from pybm.estimate.integrate import SingleShootingEstimator

def true_x(t):
    return np.exp(-t/2) + 1 

# optimal parameters are c1=-0.5, c2=0.5
def f(t, x, c1, c2):
    return c1 * x + c2

T = np.linspace(0, 10, 100)
X = true_x(T)

estimator = SingleShootingEstimator(f)

results = estimator.fit(T, X, initial_guess=(1, 1))
c1, c2 = results.x
print(f"Estimated parameters: c1={c1}, c2={c2}")
print(f"Estimated equation: x'(t) = {c1} * x(t) + {c2}")
print(f"True equation: x'(t) = -0.5 * x(t) + 0.5")


Estimated parameters: c1=-0.4999998695682803, c2=0.4999998022258442
Estimated equation: x'(t) = -0.4999998695682803 * x(t) + 0.4999998022258442
True equation: x'(t) = -0.5 * x(t) + 0.5
